In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import re 
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn import metrics
from sklearn.metrics import r2_score
from scipy.stats import pearsonr  
from scipy.signal import resample_poly


%matplotlib inline

In [ ]:
# Automatically retrieve folder names.
def get_subfolders(folder_path):
    """Return the names of all subfolders in the specified folder."""
    if not os.path.exists(folder_path):
        raise FileNotFoundError(f"Folder does not exist: {folder_path}")
    subfolders = [
        name for name in os.listdir(folder_path)
        if os.path.isdir(os.path.join(folder_path, name))
    ]
    return subfolders

# Define the Min-Max normalization function.
def min_max_normalize(df, feature_range=(0, 1)):
    """Normalize each DataFrame column to the specified range."""
    df_normalized = df.copy()
    min_val, max_val = feature_range
    for col in df.columns:
        col_min = df[col].min()
        col_max = df[col].max()
        if col_max == col_min:
            df_normalized[col] = min_val
        else:
            df_normalized[col] = (df[col] - col_min) * (max_val - min_val) / (col_max - col_min) + min_val
    return df_normalized

def downsample_data(df, source_fs=25):
    if source_fs not in (20, 25, 30):
        raise ValueError("source_fs must be 20, 25, or 30")

    columns = ["unit_id", "t", "value"]
    results = []

    for unit_id, group in df.sort_values(["unit_id", "t"]).groupby(
        "unit_id", sort=False, observed=True
    ):
        if source_fs in (20, 30):
            # 20 Hz: average every 2 frames; 30 Hz: average every 3 frames.
            factor = source_fs // 10
            group_index = (group["t"] - group["t"].min()) // factor
            sampled = group.groupby(group_index)["value"].mean().to_numpy()

        else:
            # 25 Hz: apply an anti-aliasing FIR filter and resample by a factor of 2/5.
            times = group["t"].to_numpy(dtype=float)
            values = group["value"].to_numpy(dtype=float)

            if not (np.isfinite(times).all() and np.isfinite(values).all()):
                raise ValueError(
                    f"unit_id={unit_id}: t or value contains NaN or infinite values"
                )

            if not np.allclose(np.diff(times), 1.0, rtol=1e-5, atol=1e-8):
                raise ValueError(
                    f"unit_id={unit_id}: Original frame indices are nonconsecutive or contain duplicates"
                )

            sampled = (
                resample_poly(values, up=2, down=5, padtype="line")
                if len(values) > 1
                else values.copy()
            )

        # For each unit_id, frame indices start independently at 0 and increase by 1.
        result = pd.DataFrame({
            "t": np.arange(len(sampled)),
            "value": sampled,
        })
        result.insert(0, "unit_id", unit_id)
        results.append(result)

    if not results:
        return df.loc[:, columns].iloc[:0].copy()

    return pd.concat(results, ignore_index=True).loc[:, columns]

In [ ]:

path_signal = r'path to your path_signal fold'  # Input folder with one subfolder per mouse containing save_result/C.csv, save_result/YrA.csv, and Area_Change.csv.
savepath = r'path to your savepath fold'  # Output folder for prediction tables, fitted curves, model results, and figures.
fig_path = savepath

# ============================================================
# Select the mouse folders to process (names must match the data folders).
# Set to None to process all folders, or use a list to process only the specified folders.
# Example: selected_folders = ['mouse 001', 'mouse 003', 'mouse 005'] or selected_folders = None
# CamkII TST 20Hz mouse 001, mouse 002, mouse 003, mouse 004, mouse 006, mouse 007, mouse 008, mouse 009, mouse 010
# Hsyn 20Hz mouse 001, mouse 002, mouse 003, mouse 004, mouse 005
# PV 20Hz mouse 001, mouse 002, mouse 003, mouse 004, mouse 007
# VIP 20Hz mouse 001, mouse 002, mouse 003, mouse 004, mouse 006
# VIP 30Hz mouse 005, mouse 007, mouse 008
# SST 30Hz mouse 004, mouse 005
# PV 25Hz mouse 005, mouse 006
# CamkII TST 25Hz mouse 005

selected_folders = ['mouse 005']  # <-- Modify as needed.
# ============================================================

subfolders = get_subfolders(path_signal)

if selected_folders is not None:
    subfolders = [f for f in subfolders if f in selected_folders]
    print(f'Selected folders: {subfolders}')

for folder in subfolders:
    folder_path = os.path.join(path_signal, folder)
    # Store results and figures in a dedicated random forest subdirectory.
    savefolder = os.path.join(savepath, folder, 'RF_regression')
    if not os.path.exists(savefolder):
        os.makedirs(savefolder)
    fig_path = savefolder

    # Read the data.
    df_signal = pd.read_csv(folder_path+'\\save_result\\C.csv')
    df_raw = pd.read_csv(folder_path+'\\save_result\\YrA.csv')
    df_behaviour = pd.read_csv(folder_path+'\\Area_Change.csv')
    
    # Standardize column names.
    df_signal = df_signal.rename(columns={"unit_id":"unit_id","frame":"t","C":"value"})
    df_raw = df_raw.rename(columns={"unit_id":"unit_id","frame":"t","YrA":"value"})

    # Step 1: Manually define the list of IDs to exclude.
    drop_list = []  # <-- Manually identified anomalous IDs; modify as needed.

    # Step 2: Identify valid IDs using C as the reference.
    valid_ids = set(df_signal['unit_id'].unique()) - set(drop_list)

    # Step 3: Retain valid IDs in YrA and C.
    df_raw = df_raw[df_raw['unit_id'].isin(valid_ids)].copy()
    df_signal = df_signal[df_signal['unit_id'].isin(valid_ids)].copy()

    # Step 4: Reassign consecutive IDs.
    new_id_map = {old_id: new_id for new_id, old_id in enumerate(sorted(valid_ids))}
    df_raw['unit_id'] = df_raw['unit_id'].map(new_id_map)
    df_signal['unit_id'] = df_signal['unit_id'].map(new_id_map)

# ============================================================
    df_raw = downsample_data(df_raw, source_fs=25)
    df_signal = downsample_data(df_signal, source_fs=25)
# ============================================================

    # Standardize C.
    df = pd.merge(df_raw, df_signal, on=['unit_id', 't'], suffixes=('_YrA', '_C'))
    df['residual'] = df['value_YrA'] - df['value_C']
    df['value_z'] = df['value_C'] / df.groupby('unit_id')['residual'].transform('std')

    # Truncate the behavioral and signal data to the shortest frame count.
    df_behaviour = df_behaviour.T.reset_index(drop=True)
    min_len = min(df.groupby('unit_id').size().min(), len(df_behaviour))

    df_signal = df.groupby('unit_id').apply(lambda g: g.sort_values('t').head(min_len)).reset_index(drop=True)
    resized_df_signal_dic_update = df_signal.pivot(index='t', columns='unit_id', values='value_z').reset_index(drop=True)
    df_behaviour_update = df_behaviour[:min_len]

    # Normalize both DataFrames.
    df_signal_normalized = min_max_normalize(resized_df_signal_dic_update)
    df_behaviour_normalized = min_max_normalize(df_behaviour_update)

    # Use a strictly chronological split to prevent temporal leakage (shuffle=False).
    S_train_raw, S_test_raw, B_train_raw, B_test_raw = train_test_split(df_signal_normalized,
                                                                        df_behaviour_normalized,
                                                                        test_size=0.2,
                                                                        random_state=0,
                                                                        shuffle=False)
    
    # Clean the training and test matrices to prevent premature cross-contamination in the main pipeline.
    S_train = S_train_raw.reset_index(drop=True)
    S_test = S_test_raw.reset_index(drop=True)
    B_train = B_train_raw.reset_index(drop=True)
    B_test = B_test_raw.reset_index(drop=True)
    
    # Flatten the ground-truth test labels for Pearson correlation calculations.
    B_test_flat = B_test.values.flatten()

    # =========================================================================
    # Main statistical pipeline: 100 experimental random forest regression runs versus 100 shuffle control runs.
    # =========================================================================
    
    # Run configuration
    ITERATIONS = 100         # Number of repeated runs (100 each for the experimental and control groups).
    RF_ESTIMATORS = 200      # 🌲 Number of decision trees per model. Setting this to 200 achieves full statistical convergence while greatly accelerating repeated fitting.
    
    # Initialize lists for experimental group metrics.
    experimental_r2_scores = []
    experimental_r_values = []
    experimental_p_values = []
    
    # Initialize lists for control group metrics.
    shuffle_r2_scores = []
    shuffle_r_values = []
    shuffle_p_values = []

    # -------------------------------------------------------------------------
    # Part 1: 100 bootstrap perturbation evaluations for the experimental group.
    # -------------------------------------------------------------------------
    print(f"\n>>> [{folder}] Starting module one: running {ITERATIONS} feature/perturbation decoding evaluations for the random forest experimental group...")
    for i in range(ITERATIONS):
        # Vary random_state across runs to introduce random bootstrap perturbations in subsample and feature selection during tree construction.
        regressor_exp = RandomForestRegressor(n_estimators=RF_ESTIMATORS, random_state=i, n_jobs=-1)
        regressor_exp.fit(S_train, B_train.values.ravel())
        
        # Generate predictions for the held-out test set.
        B_pred_exp = regressor_exp.predict(S_test)
        
        # Compute the three primary metrics.
        r2 = r2_score(B_test, B_pred_exp)
        r_val, p_val = pearsonr(B_test_flat, B_pred_exp)
        
        experimental_r2_scores.append(r2)
        experimental_r_values.append(r_val)
        experimental_p_values.append(p_val)
        
        if (i + 1) % (ITERATIONS // 5) == 0:
            print(f"    -> Experimental group progress: {i + 1}/{ITERATIONS} completed.")

    # -------------------------------------------------------------------------
    # Part 2: 100 permutation evaluations with independent shuffling of both inputs and targets for the shuffle control group.
    # -------------------------------------------------------------------------
    print(f"\n>>> [{folder}] Starting module two: running {ITERATIONS} independent permutation trials with both inputs and targets shuffled for the random forest control group...")
    for i in range(ITERATIONS):
        # Apply independent frame-level permutations to disrupt the alignment between spatial feature patterns and behavioral variation.
        s_shuffle_indices = np.random.permutation(len(S_train))
        b_shuffle_indices = np.random.permutation(len(B_train))
        
        S_train_shuffled = S_train.iloc[s_shuffle_indices].reset_index(drop=True)
        B_train_shuffled = B_train.iloc[b_shuffle_indices].reset_index(drop=True)
        
        regressor_shuf = RandomForestRegressor(n_estimators=RF_ESTIMATORS, random_state=i, n_jobs=-1)
        regressor_shuf.fit(S_train_shuffled, B_train_shuffled.values.ravel())
        
        # Evaluate the baseline generalization performance on the original held-out test set with temporal order preserved and no shuffling.
        B_pred_shuf = regressor_shuf.predict(S_test)
        
        # Compute the three primary metrics.
        r2_shuf = r2_score(B_test, B_pred_shuf)
        r_val_shuf, p_val_shuf = pearsonr(B_test_flat, B_pred_shuf)
        
        shuffle_r2_scores.append(r2_shuf)
        shuffle_r_values.append(r_val_shuf)
        shuffle_p_values.append(p_val_shuf)
        
        if (i + 1) % (ITERATIONS // 5) == 0:
            print(f"    -> Control group progress: {i + 1}/{ITERATIONS} completed.")

    # Compute summary means.
    print(f"\n    -> [RF summary] Experimental group mean R²: {np.mean(experimental_r2_scores):.4f}, mean r: {np.mean(experimental_r_values):.4f}")
    print(f"    -> [RF summary] Control group mean R²: {np.mean(shuffle_r2_scores):.4f}, mean r: {np.mean(shuffle_r_values):.4f}")

    # -------------------------------------------------------------------------
    # Part 3: Panel A: R² distributions; Panel B: Pearson r distributions.
    # -------------------------------------------------------------------------
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # Assemble a long-format DataFrame for plotting.
    df_plot = pd.DataFrame({
        'R2_Score': experimental_r2_scores + shuffle_r2_scores,
        'Pearson_r': experimental_r_values + shuffle_r_values,
        'Group': ['Experimental\n(Actual)'] * ITERATIONS + ['Shuffle\n(Control)'] * ITERATIONS
    })
    
    # --- Left panel: compare distributions of the coefficient of determination (R2). ---
    sns.boxplot(x='Group', y='R2_Score', data=df_plot, palette=['#4C72B0', '#C44E52'], 
                width=0.4, showfliers=False, boxprops=dict(alpha=0.3), ax=axes[0])
    sns.stripplot(x='Group', y='R2_Score', data=df_plot, palette=['#4C72B0', '#C44E52'], 
                  alpha=0.4, jitter=0.25, size=2.5, ax=axes[0])
    axes[0].axhline(y=np.mean(experimental_r2_scores), color='#4C72B0', linestyle='--', linewidth=1.5, label='Mean Exp')
    axes[0].axhline(y=np.mean(shuffle_r2_scores), color='#C44E52', linestyle='--', linewidth=1.5, label='Mean Shuf')
    axes[0].axhline(y=0, color='black', linestyle=':', alpha=0.5)
    axes[0].set_title('Random Forest Decoding Accuracy ($R^2$ Score)', fontsize=11, fontweight='bold')
    axes[0].set_ylabel('$R^2$ Score', fontsize=11)
    axes[0].grid(True, axis='y', linestyle=':', alpha=0.6)
    axes[0].legend(loc='best')
    
    # --- Right panel: compare distributions of Pearson correlation coefficients. ---
    sns.boxplot(x='Group', y='Pearson_r', data=df_plot, palette=['#4C72B0', '#C44E52'], 
                width=0.4, showfliers=False, boxprops=dict(alpha=0.3), ax=axes[1])
    sns.stripplot(x='Group', y='Pearson_r', data=df_plot, palette=['#4C72B0', '#C44E52'], 
                  alpha=0.4, jitter=0.25, size=2.5, ax=axes[1])
    axes[1].axhline(y=np.mean(experimental_r_values), color='#4C72B0', linestyle='--', linewidth=1.5, label='Mean Exp')
    axes[1].axhline(y=np.mean(shuffle_r_values), color='#C44E52', linestyle='--', linewidth=1.5, label='Mean Shuf')
    axes[1].axhline(y=0, color='black', linestyle=':', alpha=0.5)
    axes[1].set_title('Random Forest Phase Tracking (Pearson\'s $r$)', fontsize=11, fontweight='bold')
    axes[1].set_ylabel('Pearson Correlation Coefficient ($r$)', fontsize=11)
    axes[1].grid(True, axis='y', linestyle=':', alpha=0.6)
    axes[1].legend(loc='best')
    
    plt.suptitle(f'Random Forest Regressor Verification & Permutation Test\n(Dataset: Hsyn | Folder: {folder})', fontsize=12, fontweight='bold', y=0.98)
    plt.tight_layout()
    plt.savefig(os.path.join(savefolder, 'rf_comprehensive_metrics_comparison.png'), dpi=300)
    plt.show()
    plt.close()
    
    # -------------------------------------------------------------------------
    # Part 4: save all quantitative metrics (200 rows containing R2, r, and p-value).
    # -------------------------------------------------------------------------
    df_metrics_output = pd.DataFrame({
        'iteration': range(1, ITERATIONS + 1),
        'experimental_r2_score': experimental_r2_scores,
        'experimental_r_value': experimental_r_values,
        'experimental_p_value': experimental_p_values,
        'shuffle_r2_score': shuffle_r2_scores,
        'shuffle_r_value': shuffle_r_values,
        'shuffle_p_value': shuffle_p_values
    })
    df_metrics_output.to_csv(os.path.join(savefolder, 'rf_full_statistical_distributions.csv'), index=False)
    
    print(f">>> [{folder}] The two-panel figure and data table containing all random forest metrics have been saved to: {savefolder}\n" + "="*60)